# Regular expressions with the `re` module

This notebook practices regular expressions on small texts, on a connection log, then on a corpus of nearly 4,000 fraudulent e-mails:

1. Validating inputs
2. Phone numbers in one format
3. HTML links to Markdown
4. Cleaning up a text copied from a PDF
5. Quoted strings
6. Intrusion attempts
7. A corpus of fraudulent e-mails
8. Cleaning an accented text (optional)

The exercises are independent, except the steps of exercise 7, which build on each other. Each exercise has a cell for your code, then additional hints and its solution, collapsed: open them if needed.

To work out a pattern, https://regex101.com/ (with the "Python" flavor) explains each of its parts and shows what it matches. If you need a refresher on the basics, the https://regexone.com/ tutorial goes through them step by step.

## 1. Validating inputs

A French licence plate is written `AB-123-CD`: two capital letters, a hyphen, three digits, a hyphen, two capital letters, that is the pattern `pattern` below.

For each string of `plates`, display whether `re.search` and `re.fullmatch` find the pattern in it (their result is `None` or a `Match` object). You should get:

```
AB-123-CD    search: True  fullmatch: True
ab-123-cd    search: False fullmatch: False
xAB-123-CDx  search: True  fullmatch: False
AB-1234-CD   search: False fullmatch: False
AB-123-CDE   search: True  fullmatch: False
GH-007-XY    search: True  fullmatch: True
```

Which of the two functions validates an input? Then, with `re.compile` and the `re.IGNORECASE` flag, build `valid`, a compiled pattern that also accepts lowercase letters, and display the list of the plates `valid.fullmatch` accepts: `['AB-123-CD', 'ab-123-cd', 'GH-007-XY']`.

In [ ]:
import re

plates = [
  "AB-123-CD",
  "ab-123-cd",
  "xAB-123-CDx",
  "AB-1234-CD",
  "AB-123-CDE",
  "GH-007-XY",
]
pattern = r"[A-Z]{2}-\d{3}-[A-Z]{2}"

# Your code here

### Additional hints

- `re.search(pattern, plate) is not None` is `True` when the pattern is found anywhere in `plate`; `re.fullmatch`, only when it covers the whole string
- in an f-string, `{plate:12}` pads `plate` with spaces up to 12 characters, and `{s!s:5}` does the same for the boolean `s`
- `re.compile(pattern, re.IGNORECASE)` returns a compiled pattern, whose `fullmatch(plate)` method is used like `re.fullmatch`

A skeleton to copy into the exercise cell, replacing each `...`:

```python
for plate in plates:
  s = ... is not None
  f = ... is not None
  print(f"{plate:12} search: {s!s:5} fullmatch: {f}")

valid = re.compile(pattern, ...)
print([plate for plate in plates if ...])
```

### Solution

In [ ]:
for plate in plates:
  s = re.search(pattern, plate) is not None
  f = re.fullmatch(pattern, plate) is not None
  print(f"{plate:12} search: {s!s:5} fullmatch: {f}")

# re.fullmatch validates an input: re.search also accepts a plate surrounded by
# other characters (xAB-123-CDx, AB-123-CDE).

valid = re.compile(pattern, re.IGNORECASE)
print([plate for plate in plates if valid.fullmatch(plate)])

## 2. Phone numbers in one format

A form received the French phone numbers of `numbers`, written in six ways. Write a function `normalize(number)` that returns the number in the format `06 39 98 12 34`: ten digits in groups of two, the `+33` country code replaced with a `0`. Then display the normalized number of each item of `numbers`. You should get:

```
06 39 98 12 34
06 39 98 12 34
06 39 98 12 34
06 39 98 12 34
01 99 00 56 78
01 99 00 56 78
```

In [ ]:
numbers = [
  "06 39 98 12 34",
  "06.39.98.12.34",
  "0639981234",
  "+33 6 39 98 12 34",
  "01-99-00-56-78",
  "+33 (0)1 99 00 56 78",
]

# Your code here

### Additional hints

Three replacements in `normalize`:

- `\D` matches what isn't a digit: removing it keeps only the digits, and `"+33 (0)1 99 00 56 78"` becomes `"330199005678"`
- a `33` at the start of the string, followed or not by a `0`, becomes a `0`: `^` anchors the pattern at the start, `?` makes the `0` optional
- five groups of two digits, `(\d{2})` five times, are rewritten `\1 \2 \3 \4 \5`

Example: `normalize("+33 (0)1 99 00 56 78")` returns `"01 99 00 56 78"`.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def normalize(number: str) -> str:
  digits = re.sub(r"\D", "", number)
  digits = re.sub(r"...", "0", digits)
  return re.sub(r"...", r"\1 \2 \3 \4 \5", digits)


for number in numbers:
  print(normalize(number))
```

### Solution

In [ ]:
def normalize(number: str) -> str:
  digits = re.sub(r"\D", "", number)
  digits = re.sub(r"^330?", "0", digits)
  return re.sub(r"(\d{2})(\d{2})(\d{2})(\d{2})(\d{2})", r"\1 \2 \3 \4 \5", digits)


for number in numbers:
  print(normalize(number))

## 3. HTML links to Markdown

In HTML, a link is written `<a href="address">text</a>`; in Markdown, `[text](address)`. With `re.sub`, rewrite the two links of `html` in Markdown, and display the result:

```
<p>See [the re docs](https://docs.python.org/3/library/re.html) and [regex101](https://regex101.com/).</p>
```

First try the pattern `<a href="(.*)">(.*)</a>`: why doesn't it give this result? Fix it.

In [ ]:
html = (
  '<p>See <a href="https://docs.python.org/3/library/re.html">the re docs</a>'
  ' and <a href="https://regex101.com/">regex101</a>.</p>'
)

# Your code here

### Additional hints

The replacement reuses the groups in the other order: `r"[\2](\1)"`.

`*` is greedy: it repeats as much as it can, and the first `(.*)` runs up to the last `">` of the line, across the first link. Two fixes:

- forbid the double quote in the address and the angle bracket in the text, with negated character classes: `[^"]*` and `[^<]*`
- or make the repetition non-greedy, `.*?`, which repeats as little as it can

A skeleton to copy into the exercise cell, replacing each `...`:

```python
print(re.sub(r'<a href="(...)">(...)</a>', r"[\2](\1)", html))
```

### Solution

In [ ]:
# With negated character classes
print(re.sub(r'<a href="([^"]*)">([^<]*)</a>', r"[\2](\1)", html))
# With non-greedy repetitions
print(re.sub(r'<a href="(.*?)">(.*?)</a>', r"[\2](\1)", html))

## 4. Cleaning up a text copied from a PDF

The text `text`, copied from a PDF, has extra blanks: several spaces or a tab between two words, a space before commas and periods, spaces at the end of lines. With `re.sub`, build `clean`, the text without these blanks, and display it. You should get:

```
The re module searches for patterns.
It replaces, splits and extracts
the pieces of a text.
```

Keep the line breaks: only the spaces and tabs are extra.

In [ ]:
import re

text = (
  "The  re module   searches for patterns .\n"
  "It\treplaces , splits and extracts   \n"
  "the pieces of a text ."
)

# Your code here

### Additional hints

Three replacements, one after the other:

- `[ \t]+` matches a run of spaces and tabs, to replace with a single space. `\s+` would go too far: it also matches the line break `\n`, and would join the lines
- a space followed by a comma or a period: capture the punctuation in a group, and put it back alone with `\1` in the replacement
- the spaces right before the end of a line: `$` matches every end of line only with the `re.MULTILINE` flag, passed to `re.sub` by name, `flags=`

A skeleton to copy into the exercise cell, replacing each `...`:

```python
clean = re.sub(r"[ \t]+", " ", text)
clean = re.sub(r"...", r"\1", clean)
clean = re.sub(r"...", "", clean, flags=...)
print(clean)
```

### Solution

In [ ]:
clean = re.sub(r"[ \t]+", " ", text)
clean = re.sub(r" ([,.])", r"\1", clean)
clean = re.sub(r" +$", "", clean, flags=re.MULTILINE)
print(clean)

## 5. Quoted strings

In Python, a string is written between `"` or between `'`, and each can contain the other quote. With `re.finditer`, display the content of each string of the line of code `line`, one per line. You should get:

```
Hello
everyone
it's
he said "yes"
```

The pattern `["'](.*?)["']` doesn't work: it stops at the apostrophe of `it's`. Capture the opening quote in a group, and require the same one to close with a backreference.

In [ ]:
line = """print("Hello", 'everyone', "it's", 'he said "yes"')"""

# Your code here

### Additional hints

- in a pattern, `\1` matches exactly the text the first group captured: `(["'])` captures the opening quote, and `\1` requires the same one to close
- the string's content is the second group, `match.group(2)`, a non-greedy repetition `.*?` that stops at the first closing quote (see exercise 3)

A skeleton to copy into the exercise cell, replacing each `...`:

```python
for match in re.finditer(r"""...""", line):
  print(...)
```

### Solution

In [ ]:
for match in re.finditer(r"""(["'])(.*?)\1""", line):
  print(match.group(2))

## 6. Intrusion attempts

The next cell writes `auth.log`, a server's connection log: each line gives the date, the host, the service and its process number, then a message. A failed SSH login attempt reads:

```
Oct 12 10:30:02 web01 sshd[3118]: Failed password for root from 203.0.113.7 port 52144 ssh2
Oct 12 10:31:57 web01 sshd[3120]: Failed password for invalid user admin from 203.0.113.7 port 52160 ssh2
```

`invalid user` comes before an account that doesn't exist on the server. Read `auth.log` into `log`. With `re.finditer` and named groups `user` and `ip`, count the failures of each address in the dictionary `failures`, and store the accounts each address targets in the dictionary of sets `accounts`. Finally, in the order of their first failure, display the addresses with at least three failures, with their number of failures and the sorted list of the accounts they target. You should get:

```
203.0.113.7: 7 failures, accounts ['admin', 'oracle', 'root']
198.51.100.23: 4 failures, accounts ['git', 'root', 'ubuntu']
```

In [ ]:
import re
from pathlib import Path

Path("auth.log").write_text("""\
Oct 12 10:30:02 web01 sshd[3118]: Failed password for root from 203.0.113.7 port 52144 ssh2
Oct 12 10:30:05 web01 sshd[3118]: Failed password for root from 203.0.113.7 port 52144 ssh2
Oct 12 10:30:09 web01 sshd[3118]: Failed password for root from 203.0.113.7 port 52144 ssh2
Oct 12 10:30:11 web01 sshd[3118]: error: maximum authentication attempts exceeded for root from 203.0.113.7 port 52144 ssh2 [preauth]
Oct 12 10:31:57 web01 sshd[3120]: Failed password for invalid user admin from 203.0.113.7 port 52160 ssh2
Oct 12 10:32:03 web01 sshd[3120]: Failed password for invalid user admin from 203.0.113.7 port 52160 ssh2
Oct 12 10:33:40 web01 sshd[3125]: Failed password for invalid user oracle from 203.0.113.7 port 52188 ssh2
Oct 12 10:45:00 web01 CRON[3190]: pam_unix(cron:session): session opened for user root by (uid=0)
Oct 12 10:45:01 web01 CRON[3190]: pam_unix(cron:session): session closed for user root
Oct 12 11:02:13 web01 sshd[3241]: Failed password for carole from 192.0.2.15 port 40102 ssh2
Oct 12 11:02:19 web01 sshd[3241]: Accepted password for carole from 192.0.2.15 port 40102 ssh2
Oct 12 11:02:19 web01 sshd[3241]: pam_unix(sshd:session): session opened for user carole by (uid=0)
Oct 12 11:12:40 web01 sshd[3302]: Accepted publickey for bruno from 192.0.2.20 port 40112 ssh2
Oct 12 11:12:40 web01 sshd[3302]: pam_unix(sshd:session): session opened for user bruno by (uid=0)
Oct 12 11:20:31 web01 sshd[3330]: Invalid user ubuntu from 198.51.100.23 port 51000
Oct 12 11:20:33 web01 sshd[3330]: Failed password for invalid user ubuntu from 198.51.100.23 port 51000 ssh2
Oct 12 11:20:38 web01 sshd[3330]: Failed password for invalid user ubuntu from 198.51.100.23 port 51000 ssh2
Oct 12 11:21:02 web01 sshd[3333]: Failed password for root from 198.51.100.23 port 51012 ssh2
Oct 12 11:21:05 web01 sshd[3333]: Connection closed by authenticating user root 198.51.100.23 port 51012 [preauth]
Oct 12 11:48:17 web01 sshd[3360]: Failed password for invalid user test from 203.0.113.99 port 60122 ssh2
Oct 12 11:48:21 web01 sshd[3360]: Failed password for invalid user test from 203.0.113.99 port 60122 ssh2
Oct 12 11:48:22 web01 sshd[3360]: Disconnected from invalid user test 203.0.113.99 port 60122 [preauth]
Oct 12 12:03:44 web01 sshd[3412]: Failed password for root from 203.0.113.7 port 52301 ssh2
Oct 12 12:15:09 web01 sshd[3415]: Received disconnect from 192.0.2.20 port 40112:11: disconnected by user
Oct 12 12:15:09 web01 sshd[3415]: pam_unix(sshd:session): session closed for user bruno
Oct 12 12:30:00 web01 sudo[3501]:   carole : TTY=pts/0 ; PWD=/home/carole ; USER=root ; COMMAND=/usr/bin/apt update
Oct 12 13:05:51 web01 sshd[3577]: Failed password for invalid user git from 198.51.100.23 port 51340 ssh2
Oct 12 13:06:02 web01 sshd[3580]: Accepted publickey for carole from 192.0.2.15 port 40250 ssh2
""")

# Your code here

### Additional hints

- `Path("auth.log").read_text()` returns the file's content
- `Failed password for (?:invalid user )?(?P<user>\S+) from (?P<ip>\S+)` matches both forms of failure: the non-capturing group makes `invalid user ` optional without adding a group. The other lines, accepted logins or `error: maximum authentication attempts exceeded`, aren't password failures
- `match["ip"]` and `match["user"]` give the text of the named groups
- `failures.get(ip, 0)` is `0` for an address not seen yet; a dictionary keeps the insertion order of its keys, hence the order of the first failures
- `sorted(a_set)` returns the sorted list of a set's elements

A skeleton to copy into the exercise cell, replacing each `...`:

```python
log = Path("auth.log").read_text()
failures = {}
accounts = {}
for match in re.finditer(r"...", log):
  ip = ...
  failures[ip] = failures.get(ip, 0) + 1
  if ip not in accounts:
    accounts[ip] = set()
  accounts[ip].add(...)

for ip, count in failures.items():
  if count >= 3:
    print(f"{ip}: {count} failures, accounts {sorted(accounts[ip])}")
```

### Solution

In [ ]:
log = Path("auth.log").read_text()
failures = {}
accounts = {}
for match in re.finditer(
  r"Failed password for (?:invalid user )?(?P<user>\S+) from (?P<ip>\S+)", log
):
  ip = match["ip"]
  failures[ip] = failures.get(ip, 0) + 1
  if ip not in accounts:
    accounts[ip] = set()
  accounts[ip].add(match["user"])

for ip, count in failures.items():
  if count >= 3:
    print(f"{ip}: {count} failures, accounts {sorted(accounts[ip])}")

## 7. A corpus of fraudulent e-mails

The CLAIR corpus gathers 3,976 "Nigerian" scam e-mails (419 fraud), received from 1998 to 2007, one after the other in a single file. The file `debug_file.txt` holds only two of them, to work out the code. Source: Radev (2008), *CLAIR collection of fraud email*, ACL Data and Code Repository, ADCR2008T001, published on https://www.kaggle.com/datasets/rtatman/fraudulent-email-corpus under the CC BY-SA 4.0 license.

Each e-mail starts with a `From r` line, followed by its headers, including:

- `From`: the sender (sometimes blank)
- `Reply-To`: the address to reply to
- `To`: the recipient (sometimes truncated for anonymity)
- `Date`: the date it was sent
- `Subject`: the message's subject
- `Content-Type`: the content's type and its encoding

and finally, after a blank line, the message's body.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh fraud-emails

The files are encoded in `windows-1252`: the next cell reads them into `debug_text` and `work_text`, and displays the start of the corpus.

In [ ]:
import re
from pathlib import Path

debug_text = Path("fraud-emails/debug_file.txt").read_text(encoding="windows-1252")
work_text = Path("fraud-emails/fradulent_emails.txt").read_text(encoding="windows-1252")
print(work_text[:1500])

### 7.1 Splitting the corpus into e-mails

Write `split_emails(text)`, which returns the list of the e-mails of `text`: split it with `re.split` at each line that starts with `From r`. Test it on `debug_text`, which must give 2 e-mails, then store those of `work_text` in `emails`: 3,976.

**Useful documentation**

- [`re.split`](https://docs.python.org/3/library/re.html#re.split)

In [ ]:
def split_emails(text: str) -> list[str]: ...


# Your code here

#### Additional hints

- `^` matches every start of line only with the `re.MULTILINE` flag, to pass by name, `flags=` (passing it positionally is deprecated since Python 3.13)
- the file starts with a `From r` line: the first piece `re.split` returns, what comes before it, is an empty string, to drop with `[1:]`

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def split_emails(text: str) -> list[str]:
  return re.split(r"...", text, flags=...)[1:]


print(len(split_emails(debug_text)))
emails = split_emails(work_text)
print(len(emails))
```

#### Solution

In [ ]:
def split_emails(text: str) -> list[str]:
  return re.split(r"^From r", text, flags=re.MULTILINE)[1:]


print(len(split_emails(debug_text)))
emails = split_emails(work_text)
print(len(emails))

### 7.2 The domains of the reply addresses

The `Reply-To:` header lines give the address to reply to, for instance `Reply-To: james_ngola2002@maktoob.com`.

1. With `re.findall` and a pattern with only **one** group, build `domains`, the list of the domains (what follows the `@`) of the `Reply-To:` lines of `work_text`.
2. With a pattern with **two** groups, build `pairs`, the list of the (user, domain) pairs of the same lines.

Display the number of elements and the first three of each list. You should get:

```
2558 ['maktoob.com', 'epatra.com', 'www.com']
2556 [('james_ngola2002', 'maktoob.com'), ('obong_715', 'epatra.com'), ('m_abacha03', 'www.com')]
```

What does `re.findall` return in each case?

In [ ]:
# Your code here

#### Additional hints

- a domain is a series of words separated by periods: `[\w-]+(?:\.[\w-]+)+`, where the non-capturing group `(?:...)` repeats `.word` without adding a group to the result of `re.findall`
- `^Reply-To:.*?@` runs from the start of the line to the first `@`: `.*?` is non-greedy (see exercise 3), and `flags=re.MULTILINE` makes `^` match every start of line
- the user is a series of word characters, periods and hyphens, `[\w.-]+`, right before the `@`; with a greedy `.*` instead of `.*?`, it would only keep its last character
- `a_list[:3]` keeps the first three elements of a list

A skeleton to copy into the exercise cell, replacing each `...`:

```python
domains = re.findall(r"...", work_text, flags=re.MULTILINE)
print(len(domains), domains[:3])
pairs = re.findall(r"...", work_text, flags=re.MULTILINE)
print(len(pairs), pairs[:3])
```

#### Solution

In [ ]:
domains = re.findall(
  r"^Reply-To:.*?@([\w-]+(?:\.[\w-]+)+)", work_text, flags=re.MULTILINE
)
print(len(domains), domains[:3])
pairs = re.findall(
  r"^Reply-To:.*?([\w.-]+)@([\w-]+(?:\.[\w-]+)+)", work_text, flags=re.MULTILINE
)
print(len(pairs), pairs[:3])

# With one group, re.findall returns the list of this group's texts; with two, the
# list of the tuples of their texts; never the whole matched text, the line from
# "Reply-To:". The two missing pairs: users in quotes
# ("dr..abdull.rasaq"@caramail.com), which [\w.-]+ right before @ leaves out.

### 7.3 Senders and addresses

A `From:` line often has the form `From: "Mr. Ben Suleman" <bensul2004nng@spinfinder.com>`: a name, sometimes in quotes, then an address in angle brackets.

1. Write `process_name(name)`, which returns `name` without its `'` and `"`, and without its spaces at the start and at the end: `process_name(' "Mr. Ben Suleman" ')` returns `'Mr. Ben Suleman'`.
2. Build the dictionary `senders`, which maps each sender's name, passed through `process_name`, to the set of their addresses. Keep the lines of the form `name <address>` whose address is plausible (non-blank characters, an `@`, then a domain with a period) and whose name isn't empty; skip the others, which have only an address or only a name.
3. Display the number of `From:` lines kept and skipped, the number of senders, and the number of senders with several addresses. With the pattern of the hints, you should find 3,190 lines kept, 598 skipped, and 2,407 senders, 162 of whom have several addresses.

In [ ]:
def process_name(name: str) -> str: ...


# Your code here

#### Additional hints

- in `process_name`, `re.sub` with the character class `['"]` removes the quotes, and the strings' `strip` method the spaces at the start and at the end
- one pattern for the whole line, with a group for the name and one for the address: `^From:(.*)<(\w\S*@\S+\.\w+)>$`, again with `re.MULTILINE`
- `re.finditer` gives each line matched, of which `match.group(1)` and `match.group(2)` are the name and the address
- the number of `From:` lines: the length of the list `re.findall(r"^From:", work_text, flags=re.MULTILINE)` returns

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def process_name(name: str) -> str:
  return re.sub(..., "", name).strip()


senders = {}
kept = 0
for match in re.finditer(r"...", work_text, flags=re.MULTILINE):
  name = process_name(match.group(1))
  if name == "":
    continue
  kept += 1
  if name not in senders:
    senders[name] = set()
  senders[name].add(...)

total = len(re.findall(r"^From:", work_text, flags=re.MULTILINE))
several = [name for name, addresses in senders.items() if len(addresses) > 1]
print(f"{kept} lines kept, {total - kept} skipped")
print(f"{len(senders)} senders, {len(several)} of whom have several addresses")
```

#### Solution

In [ ]:
def process_name(name: str) -> str:
  return re.sub(r"['\"]", "", name).strip()


print(repr(process_name(' "Mr. Ben Suleman" ')))

senders = {}
kept = 0
for match in re.finditer(
  r"^From:(.*)<(\w\S*@\S+\.\w+)>$", work_text, flags=re.MULTILINE
):
  name = process_name(match.group(1))
  if name == "":
    continue
  kept += 1
  if name not in senders:
    senders[name] = set()
  senders[name].add(match.group(2))

total = len(re.findall(r"^From:", work_text, flags=re.MULTILINE))
several = [name for name, addresses in senders.items() if len(addresses) > 1]
print(f"{kept} lines kept, {total - kept} skipped")
print(f"{len(senders)} senders, {len(several)} of whom have several addresses")

### 7.4 E-mails per year

For each e-mail of `emails`, find the year of its first `Date:` header with `re.search` and a named group `year`, and count the e-mails of each year in the dictionary `per_year`. Display the years in order, each with its number of e-mails, then the number of e-mails without a year. You should find 1,170 e-mails in 2006, the busiest year, and 566 without a year.

The corpus spans 1998 to 2007: where do the other years come from?

In [ ]:
# Your code here

#### Additional hints

- a named group is written `(?P<year>...)`, and `match.group("year")` gives what it captured
- in `^Date:.*?(?P<year>\d{4})`, the non-greedy repetition stops at the first four-digit number of the line: `2002` in `Date: Thu, 31 Oct 2002 05:10:00`
- `re.search` returns the first match, or `None` when the e-mail has no `Date:` header with a year
- `per_year.get(year, 0)` is `0` for a year not seen yet, and `sorted(per_year.items())` gives the `(year, count)` pairs in the order of the years

A skeleton to copy into the exercise cell, replacing each `...`:

```python
per_year = {}
missing = 0
for email in emails:
  match = re.search(r"...", email, flags=re.MULTILINE)
  if match is None:
    missing += 1
  else:
    year = ...
    per_year[year] = per_year.get(year, 0) + 1

for year, count in sorted(per_year.items()):
  print(year, count)
print("Without a year:", missing)
```

#### Solution

In [ ]:
per_year = {}
missing = 0
for email in emails:
  match = re.search(r"^Date:.*?(?P<year>\d{4})", email, flags=re.MULTILINE)
  if match is None:
    missing += 1
  else:
    year = match.group("year")
    per_year[year] = per_year.get(year, 0) + 1

for year, count in sorted(per_year.items()):
  print(year, count)
print("Without a year:", missing)

# The 15 e-mails dated 1980, 1990, 2015 or 2082 come from computers with a wrong clock
# (or from forged headers): a per-year analysis leaves them out.

## 8. Cleaning an accented text (optional)

Before analyzing a text (counting its words, comparing it to others), we *normalize* it: lowercase, without accents or punctuation. What a word character is then depends on the flags. `text` is a French sentence, full of accents.

**1.** Display the words of `text`, `re.findall(r"\w+", text)`, then the same with `flags=re.ASCII`. Which version splits the words wrongly, and why?

**2.** Write `strip_accents(text)`, which returns `text` without its accents. `unicodedata.normalize("NFD", text)` decomposes each accented letter into its base letter followed by a *combining* accent (`"é"` becomes `"e"` followed by `"\u0301"`), and the combining accents fill the range from `\u0300` to `\u036f`: remove them with `re.sub`. `print(strip_accents(text))` must display:

```
Elodie a vu « Œdipe roi » au theatre : 3 cafes, 2 cremes brulees… Ca coute 12,50 € !
```

**3.** Write `clean(text)`, which returns `text` in lowercase, without accents, each run of characters that aren't word characters replaced by a single space, and without spaces at the start or at the end. `print(clean(text))` must display:

```
elodie a vu œdipe roi au theatre 3 cafes 2 cremes brulees ca coute 12 50
```

The `œ` stayed: why? What would `œdipe` become with `re.ASCII` in step 3?

In [ ]:
import re
import unicodedata

text = (
  "Élodie a vu « Œdipe roi » au théâtre : 3 cafés, 2 crèmes brûlées… Ça coûte 12,50 € !"
)

# Your code here

### Additional hints

- with `re.ASCII`, `\w` only matches `[a-zA-Z0-9_]`: each accented letter cuts the word
- in a pattern, `[\u0300-\u036f]` is the character class of this range: `re` itself understands the `\uXXXX` notation, even in a raw string
- `clean` chains `strip_accents`, the strings' `lower` method, `re.sub` with `\W+`, and the `strip` method

A skeleton to copy into the exercise cell, replacing each `...`:

```python
print(re.findall(r"\w+", text))
print(re.findall(r"\w+", text, flags=...))


def strip_accents(text: str) -> str:
  return re.sub(r"...", "", unicodedata.normalize("NFD", text))


def clean(text: str) -> str:
  return re.sub(r"...", " ", ...).strip()


print(strip_accents(text))
print(clean(text))
```

### Solution

In [ ]:
print(re.findall(r"\w+", text))
print(re.findall(r"\w+", text, flags=re.ASCII))
# With re.ASCII, an accented letter is no longer a word character: "Élodie"
# becomes "lodie", and "théâtre" is cut into "th" and "tre".


def strip_accents(text: str) -> str:
  return re.sub(r"[\u0300-\u036f]", "", unicodedata.normalize("NFD", text))


def clean(text: str) -> str:
  return re.sub(r"\W+", " ", strip_accents(text).lower()).strip()


print(strip_accents(text))
print(clean(text))
# "œ" is a ligature, not an accented letter: NFD doesn't decompose it, it has to
# be replaced by hand (text.replace("œ", "oe")). With re.ASCII, \W would take it
# for a separator: "œdipe" would become "dipe".